<a href="https://colab.research.google.com/github/Sasindu99-ai/Statistical-Learning-e22445/blob/main/Assignments/Assignment%204%20-%20Data%20Wrangling/e22445_data_wrangling_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠️ Introduction to the `data-analysis-tool` Package

Welcome to this demonstration of the **`data-analysis-tool`**, a custom Python package designed to streamline the most tedious aspects of the data science lifecycle: wrangling, preprocessing, and exploratory data analysis (EDA).

Traditional data cleaning often requires writing dozens of lines of repetitive pandas code. This package solves that by introducing a **modular pipeline architecture**, allowing us to define cleaning rules declaratively.

### Key Capabilities:
* **Automated Inspection:** Instantly generate summaries of dataset health, missing values, and data types.
* **Declarative Pipelines:** Chain together sanitizers, type correctors, and missing value handlers in a single, readable block.
* **Built-in Normalization:** Extract, scale (e.g., MinMax), and encode (e.g., One-Hot) numeric and categorical data with single function calls.
* **Rapid Visualization:** Generate univariate distributions, bivariate relationships, and correlation heatmaps dynamically based on the dataset's state.

Let's begin by installing the package directly from its repository.

In [1]:
import warnings
warnings.filterwarnings('ignore') # Optional: keeps the notebook clean from deprecation warnings

# Suppress output using -q for a cleaner notebook presentation
!pip install git+https://github.com/Sasindu99-ai/data-analysis-tool.git -q --no-cache

print("Package successfully installed!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Package successfully installed!


## 📖 Package Usage & Architecture

Before writing the code, it's helpful to understand the core components of the package that we will be using throughout this notebook:

1. **`Workspace`**: Defines the environment. We will use `Workspace.LOCAL` to read files directly from our local directory.
2. **`DataInspector`**: The central engine of the package. It acts as a wrapper around our pandas DataFrame, exposing methods to load data, run summaries, execute the preprocessing pipeline, and extract normalized data.
3. **`Pipeline Components`**: A suite of modular classes (like `MissingValueSanitizer`, `AutoTypeCorrector`, and `MissingValueHandler`) that are passed into the `DataInspector.pipeline()` method. They execute sequentially.
4. **`PlottingMethods`**: A dedicated visualization class that takes a DataFrame and automatically generates matplotlib/seaborn plots tailored to the data types present.

Let's import these tools, download our Titanic dataset, and initialize our `DataInspector`.

In [16]:
import pandas as pd
from IPython.display import display
import urllib.request
import os

# Import core components from the custom package
from data_analysis import (
    Workspace, DataInspector, PlottingMethods, get_class_info,
    MissingValueSanitizer, AutoTypeCorrector, MissingValueHandler,
    OutlierHandler, DuplicateRemover, StandardizeFormator,
    RowRemover, ColumnRemover, NumericNormalizeMethod, CategoricalNormalizeMethod
)

Next, we initialize the `DataInspector`. Because we are building this notebook to run in Google Colab, we specify `Workspace.COLAB`. This tells the package to enable Colab-specific features, such as interactive file upload widgets.

In [3]:
# Initialize the DataInspector specifically for the Google Colab environment
di = DataInspector(Workspace.COLAB)

## 3. Data Ingestion
The package is flexible and allows for different data ingestion methods. Below are two options for loading the Titanic dataset. **Please choose and run only one of the options below.**

### Option A: Interactive File Upload
By calling `upload_data()` without any parameters in the `COLAB` workspace, the package will automatically generate a file upload widget. You can use this to manually browse and upload `titanic.csv` from your local machine.

In [ ]:
# OPTION A: run this cell to use the Colab upload widget
di.upload_data()

### Option B: Download via URL (Recommended for Automation)
If you want the notebook to run fully autonomously from top to bottom, you can download the dataset directly from a web repository and pass the resulting file path directly to the `upload_data()` method.

In [17]:
# OPTION B: Run this cell to automatically fetch the data from the web

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
file_path = "titanic.csv"

if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)
    print(f"Downloaded {file_path} successfully.")

# Pass the specific file path to the inspector
di.upload_data(file_path=file_path)

## 4. Loading the DataFrame and Initial Summary
Now that the data file is secured in our workspace (either via Option A or Option B), we instruct the `DataInspector` to parse the CSV into a pandas DataFrame. We will then call `.summary()` to get an immediate snapshot of the dataset's health before we begin wrangling.

In [18]:
# Load the ingested file into active memory as a DataFrame
di.load_dataframe()

# Generate an automated summary of the raw dataset
di.summary()

DATASET SUMMARY
Rows    : 891
Columns : 12
Numerical Columns   : 7
Categorical Columns : 5
Duplicate Rows      : 0

FIRST 20 ROWS


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C



COLUMN INFORMATION


,Column,Data Type,Non-Null Count,Null Count
0,PassengerId,int64,891,0
1,Survived,int64,891,0
2,Pclass,int64,891,0
3,Name,str,891,0
4,Sex,str,891,0
5,Age,float64,714,177
6,SibSp,int64,891,0
7,Parch,int64,891,0
8,Ticket,str,891,0
9,Fare,float64,891,0



MISSING VALUES


,Column,Missing Values,Missing %
0,PassengerId,0,0.00
1,Survived,0,0.00
2,Pclass,0,0.00
3,Name,0,0.00
4,Sex,0,0.00
5,Age,177,19.87
6,SibSp,0,0.00
7,Parch,0,0.00
8,Ticket,0,0.00
9,Fare,0,0.00



NUMERICAL COLUMNS
['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

CATEGORICAL COLUMNS
['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']
SUMMARY COMPLETED


{'shape': {'rows': 891, 'columns': 12},
 'numerical_columns': ['PassengerId',
  'Survived',
  'Pclass',
  'Age',
  'SibSp',
  'Parch',
  'Fare'],
 'categorical_columns': ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'],
 'duplicate_rows': np.int64(0),
 'column_info':          Column Data Type  Non-Null Count  Null Count
 0   PassengerId     int64             891           0
 1      Survived     int64             891           0
 2        Pclass     int64             891           0
 3          Name       str             891           0
 4           Sex       str             891           0
 5           Age   float64             714         177
 6         SibSp     int64             891           0
 7         Parch     int64             891           0
 8        Ticket       str             891           0
 9          Fare   float64             891           0
 10        Cabin       str             204         687
 11     Embarked       str             889           2,
 'missing_values': 

## 2. API Discovery
Before diving into the data, let's explore the plotting capabilities available within the package by fetching the methods info and displaying it as a DataFrame.

In [13]:
# DataInspector class methods info
di_methods = di.get_methods_info()
di_methods_df = pd.DataFrame(di_methods)
print("="*60)
print("DataInspector class methods:")
print("="*60)
display(di_methods_df)
print("="*60, end="\n\n")

# DataInspector related classes info
di_rel_classes = get_class_info([
    MissingValueSanitizer, AutoTypeCorrector, MissingValueHandler,
    OutlierHandler, DuplicateRemover, StandardizeFormator,
    RowRemover, ColumnRemover, NumericNormalizeMethod, CategoricalNormalizeMethod
])
di_rel_classes_df = pd.DataFrame(di_rel_classes)
print("="*60)
print("DataInspector related classes:")
print("="*60)
display(di_rel_classes_df)
print("="*60, end="\n\n")

# PlottingMethods class methods info
pm_methods = PlottingMethods(df=di.df).get_methods_info()
pm_methods_df = pd.DataFrame(pm_methods)
print("="*60)
print("PlottingMethods class methods:")
print("="*60)
display(pm_methods_df)
print("="*60, end="\n\n")

DataInspector class methods:


,method,signature,description
0,df_ready,() -> bool,Determines if the dataframe (df) has been load...
1,extract_normalized_categorical_data,(method: data_analysis.DataInspector.Categoric...,Extracts and normalizes categorical data from ...
2,extract_normalized_numeric_data,(method: data_analysis.DataInspector.NumericNo...,Extract and normalize numeric data columns fro...
3,get_methods_info,"() -> list[dict[str, str]]",Retrieves and organizes information about all ...
4,is_colab,() -> bool,Determines whether the current execution envir...
5,load_dataframe,(dataframes: pandas.DataFrame | None = None) -...,"Loads a DataFrame into the instance, either fr..."
6,merge_normalized_data,(merge_parts: list[pandas.DataFrame] | None = ...,Merge normalized numeric and categorical data ...
7,pipeline,"(steps: list[tuple[str, data_analysis.DataInsp...",Configures a sequence of processing steps as a...
8,plot_all_associations_heatmap,"(show_matrix: bool = True, show_plot: bool = T...",Generates a heatmap of associations between fe...
9,preprocess,"(y=None, verbose: bool = True, show_summary: b...",Preprocesses the data using a pipeline of proc...



DataInspector related classes:


,class,signature,description
0,MissingValueSanitizer,(custom_missing_patterns: list[str] | None = N...,Class MissingValueSanitizer\n\n Description...
1,AutoTypeCorrector,"(numeric_formats: dict[str, str] | None = None)",Class AutoTypeCorrector\n\n Description:\n ...
2,MissingValueHandler,(strategy: data_analysis.DataInspector.Strateg...,Class MissingValueHandler\n\n Description:\...
3,OutlierHandler,"(columns: list[str] | None = None, method: dat...",Class OutlierHandler\n\n Description:\n ...
4,DuplicateRemover,"(subset: list[str] | None = None, keep: data_a...",Class DuplicateRemover\n\n Description:\n ...
5,StandardizeFormator,"(columns: list[str] | None = None, case: data_...",class StandardizeFormator\n\n Description:\...
6,RowRemover,(rows: list[int] | None = None) -> None,class RowRemover\n\n Description:\n ...
7,ColumnRemover,(columns: list[str] | None = None) -> None,class ColumnRemover\n\n Description:\n ...
8,NumericNormalizeMethod,(*values),No description available
9,CategoricalNormalizeMethod,(*values),No description available



PlottingMethods class methods:


,method,signature,description
0,bar_chart,"(column: str, color: str | None = None, title:...",Generates an HTML representation of a bar char...
1,df_ready,() -> bool,Determines if the dataframe (df) has been load...
2,get_methods_info,(),Retrieves information about public methods of ...
3,histogram,"(column: str, bins: int = 30, title: str | Non...",Generates an HTML string representation of a h...
4,pie_chart,"(column: str, title: str | None = None) -> str...",Generate an HTML representation of a pie chart...
5,plot_all_associations_heatmap,(),Generates a heatmap visualizing the associatio...
6,plot_categorical_frequency,(column: str),Generates a bar chart visualizing the frequenc...
7,plot_numeric_univariate,(columns: list[str] | None = None),Generates a series of univariate visualization...
8,plot_relationship,"(x: str, y: str)",Plots a visual representation of the relations...


## 3. Building the Cleaning Pipeline
The Titanic dataset is notorious for missing values (specifically in `Age`, `Cabin`, and `Embarked`).
We will construct a pipeline to:
1. Sanitize standard missing value formats.
2. Standardize string casing (e.g., converting `Sex` to uppercase).
3. Automatically correct inferred data types.
4. Impute `Age` using the Median, `Embarked` using the Mode, and `Cabin` dynamically as "Unknown".
5. Remove any absolute duplicate records based on `PassengerId`.

In [19]:
di.pipeline([
    ('missing_value_sanitizer', MissingValueSanitizer()),
    ('standardize_formator', StandardizeFormator(
        columns=['Sex'],
        case=StandardizeFormator.cases.UPPER,
        custom_mappings={}
    )),
    ('auto_type_corrector', AutoTypeCorrector()),
    ('missing_value_handler_age', MissingValueHandler(
        strategy=MissingValueHandler.strategies.MEDIAN,
        columns=['Age']
    )),
    ('missing_value_handler_embarked', MissingValueHandler(
        strategy=MissingValueHandler.strategies.MODE,
        columns=['Embarked']
    )),
    ('missing_value_handler_cabin', MissingValueHandler(
        strategy=MissingValueHandler.strategies.CONSTANT if hasattr(MissingValueHandler.strategies, 'CONSTANT') else MissingValueHandler.strategies.MODE,
        columns=['Cabin'],
        fill_value='Unknown' if hasattr(MissingValueHandler.strategies, 'CONSTANT') else None
    )),
    ('duplicate_remover', DuplicateRemover(
        subset=['PassengerId']
    )),
    ('outlier_handler', OutlierHandler()),
    ('remove_columns', ColumnRemover(columns=['PassengerId', 'Name', 'Ticket'] if hasattr(ColumnRemover, 'columns') else None))
])

## 4. Executing Preprocessing & Diff Summary
With the pipeline configured, we'll execute the preprocessing step. Observe the changes in the dataset's integrity (e.g., missing values eliminated, specific types corrected).

In [21]:
# Run the pipeline
di.preprocess(show_summary=False)

STARTING PREPROCESSING PIPELINE

[1/9] Running: missing_value_sanitizer
MISSING VALUE SANITIZATION COMPLETED
Missing values before : 866
Missing values after  : 866
New missing detected  : 0
✓ missing_value_sanitizer completed

[2/9] Running: standardize_formator
FORMAT STANDARDIZATION COMPLETED
Columns processed : 1
['Sex']
Case format      : UPPER
✓ standardize_formator completed

[3/9] Running: auto_type_corrector
AUTO TYPE CORRECTION COMPLETED

Converted Columns: 
['Ticket']

Skipped Columns: 
['Name', 'Sex', 'Cabin', 'Embarked']

Updated Data Types: 
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket         float64
Fare           float64
Cabin              str
Embarked           str
dtype: object
✓ auto_type_corrector completed

[4/9] Running: missing_value_handler_age
MISSING VALUE IMPUTATION COMPLETED
Missing before : 1096
Missing after  : 

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,0,3,MALE,22.0,1.0,0,7.2500,Unknown,S
1,1,1,FEMALE,38.0,1.0,0,65.6344,C85,C
2,1,3,FEMALE,26.0,0.0,0,7.9250,Unknown,S
3,1,1,FEMALE,35.0,1.0,0,53.1000,C123,S
4,0,3,MALE,35.0,0.0,0,8.0500,Unknown,S
...,...,...,...,...,...,...,...,...,...
886,0,2,MALE,27.0,0.0,0,13.0000,Unknown,S
887,1,1,FEMALE,19.0,0.0,0,30.0000,B42,S
888,0,3,FEMALE,28.0,1.0,0,23.4500,Unknown,S
889,1,1,MALE,26.0,0.0,0,30.0000,C148,C


In [22]:
print("="*60)
print("POST-CLEANING DATA SUMMARY")
print("="*60)
di.summary()

POST-CLEANING DATA SUMMARY
DATASET SUMMARY
Rows    : 891
Columns : 9
Numerical Columns   : 6
Categorical Columns : 3
Duplicate Rows      : 117

FIRST 20 ROWS


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,0,3,MALE,22.0,1.0,0,7.2500,Unknown,S
1,1,1,FEMALE,38.0,1.0,0,65.6344,C85,C
2,1,3,FEMALE,26.0,0.0,0,7.9250,Unknown,S
3,1,1,FEMALE,35.0,1.0,0,53.1000,C123,S
4,0,3,MALE,35.0,0.0,0,8.0500,Unknown,S
5,0,3,MALE,28.0,0.0,0,8.4583,Unknown,Q
6,0,1,MALE,54.0,0.0,0,51.8625,E46,S
7,0,3,MALE,2.5,2.5,0,21.0750,Unknown,S
8,1,3,FEMALE,27.0,0.0,0,11.1333,Unknown,S
9,1,2,FEMALE,14.0,1.0,0,30.0708,Unknown,C



COLUMN INFORMATION


,Column,Data Type,Non-Null Count,Null Count
0,Survived,int64,891,0
1,Pclass,int64,891,0
2,Sex,str,891,0
3,Age,float64,891,0
4,SibSp,float64,891,0
5,Parch,int64,891,0
6,Fare,float64,891,0
7,Cabin,str,891,0
8,Embarked,str,891,0



MISSING VALUES


,Column,Missing Values,Missing %
0,Survived,0,0.0
1,Pclass,0,0.0
2,Sex,0,0.0
3,Age,0,0.0
4,SibSp,0,0.0
5,Parch,0,0.0
6,Fare,0,0.0
7,Cabin,0,0.0
8,Embarked,0,0.0



NUMERICAL COLUMNS
['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

CATEGORICAL COLUMNS
['Sex', 'Cabin', 'Embarked']
SUMMARY COMPLETED


{'shape': {'rows': 891, 'columns': 9},
 'numerical_columns': ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'],
 'categorical_columns': ['Sex', 'Cabin', 'Embarked'],
 'duplicate_rows': np.int64(117),
 'column_info':      Column Data Type  Non-Null Count  Null Count
 0  Survived     int64             891           0
 1    Pclass     int64             891           0
 2       Sex       str             891           0
 3       Age   float64             891           0
 4     SibSp   float64             891           0
 5     Parch     int64             891           0
 6      Fare   float64             891           0
 7     Cabin       str             891           0
 8  Embarked       str             891           0,
 'missing_values':      Column  Missing Values  Missing %
 0  Survived               0        0.0
 1    Pclass               0        0.0
 2       Sex               0        0.0
 3       Age               0        0.0
 4     SibSp               0        0.0
 5     Par

## 5. Feature Engineering: Normalization
Machine learning algorithms typically perform best when numerical data is on a similar scale. For example, in the Titanic dataset, `Fare` spans a much larger range of numbers than `Age` or `Pclass`. If left unscaled, a model might assume `Fare` is vastly more important simply because the numbers are bigger.

We will use **MinMax Scaling** to compress all numerical values into a standard range (usually 0 to 1).

In [23]:
# Extract and scale numerical features
normalized_numerical_df = di.extract_normalized_numeric_data(
    method=NumericNormalizeMethod.MINMAX
)

print("="*60)
print("NORMALIZED NUMERICAL DATA")
print("="*60)
display(normalized_numerical_df.head())

NUMERIC NORMALIZATION COMPLETED
Method used : NumericNormalizeMethod.MINMAX
Columns      : 6
['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
NORMALIZED NUMERICAL DATA


,Survived,Pclass,Age,SibSp,Parch,Fare
0,0.0,1.0,0.375000,0.4,0.0,0.110460
1,1.0,0.0,0.682692,0.4,0.0,1.000000
2,1.0,1.0,0.451923,0.0,0.0,0.120745
3,1.0,0.0,0.625000,0.4,0.0,0.809027
4,0.0,1.0,0.625000,0.0,0.0,0.122649


## 6. Feature Engineering: Categorical Encoding
Machine learning models cannot read raw text; they require numbers. We need to convert categorical variables (like `Sex` or `Embarked`) into a mathematical format.

We will use **One-Hot Encoding**, which creates a new binary column (0 or 1) for each unique category (e.g., `Sex_MALE`, `Sex_FEMALE`).

In [24]:
# Extract and encode categorical features
normalized_categorical_df = di.extract_normalized_categorical_data(
    method=CategoricalNormalizeMethod.ONEHOT,
)

print("="*60)
print("NORMALIZED CATEGORICAL DATA")
print("="*60)
display(normalized_categorical_df.head())

CATEGORICAL ENCODING COMPLETED
Method used : CategoricalNormalizeMethod.ONEHOT
Columns     : ['Cabin', 'Embarked', 'Sex']
Output shape: (891, 153)
NORMALIZED CATEGORICAL DATA


,Cabin_A10,Cabin_A14,Cabin_A16,Cabin_A19,Cabin_A20,Cabin_A23,Cabin_A24,Cabin_A26,Cabin_A31,Cabin_A32,...,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Cabin_Unknown,Embarked_C,Embarked_Q,Embarked_S,Sex_FEMALE,Sex_MALE
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,True,False,True
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,True,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,True,True,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,True,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,True,False,True


## 7. Merging the Processed Subsets
Now that our numerical data is scaled and our categorical data is encoded, we need to stitch them back together into a single, cohesive dataframe. This final dataset is now fully preprocessed, mathematically standardized, and ready to be fed into a machine learning model.

In [25]:
# Merge the subsets back into a cohesive dataset
merged_df = di.merge_normalized_data(merge_parts=[
    normalized_numerical_df,
    normalized_categorical_df
])

print("="*60)
print("FINAL MERGED & ENCODED DATA")
print("="*60)
display(merged_df.head())

FEATURE MERGING COMPLETED
Shape: (891, 159)
Numeric features : 6
Categorical features : 153
FINAL MERGED & ENCODED DATA


,Survived,Pclass,Age,SibSp,Parch,Fare,Cabin_A10,Cabin_A14,Cabin_A16,Cabin_A19,...,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Cabin_Unknown,Embarked_C,Embarked_Q,Embarked_S,Sex_FEMALE,Sex_MALE
0,0.0,1.0,0.375000,0.4,0.0,0.110460,False,False,False,False,...,False,False,False,False,True,False,False,True,False,True
1,1.0,0.0,0.682692,0.4,0.0,1.000000,False,False,False,False,...,False,False,False,False,False,True,False,False,True,False
2,1.0,1.0,0.451923,0.0,0.0,0.120745,False,False,False,False,...,False,False,False,False,True,False,False,True,True,False
3,1.0,0.0,0.625000,0.4,0.0,0.809027,False,False,False,False,...,False,False,False,False,False,False,False,True,True,False
4,0.0,1.0,0.625000,0.0,0.0,0.122649,False,False,False,False,...,False,False,False,False,True,False,False,True,False,True


## 8. Visualizing the Processed Data
Now that our dataset is fully wrangled, scaled, and encoded, it's time to explore it visually. The `PlottingMethods` class simplifies this by automatically generating appropriate charts based on the data types we pass to it.

First, let's initialize our plotter with the final `merged_df` and look at the **Univariate Distributions** of our numerical features to see how our MinMax scaling altered their shapes.

In [33]:
# Initialize the visualization engine with the final dataset
plotter = PlottingMethods(merged_df)

# 1. Univariate analysis of numeric features
print("="*60)
print("NUMERIC UNIVARIATE DISTRIBUTIONS")
print("="*60)

univariate_plots = plotter.plot_numeric_univariate()

# Display each plot generated by the method
for fig in univariate_plots.values():
    display(fig)

NUMERIC UNIVARIATE DISTRIBUTIONS


### 8.1 Categorical Frequency
Next, we can inspect our categorical data. Since we applied One-Hot Encoding, categorical states are now represented as binary columns (0 or 1). Let's visualize the frequency of a specific encoded category, such as male passengers.

In [34]:
# 2. Visualize the frequency of a specific categorical feature
print("="*60)
print("CATEGORICAL FREQUENCY: GENDER")
print("="*60)

# Note: Adjust the column string if your StandardizeFormator/OneHot output differs slightly
plotter.plot_categorical_frequency('Sex_MALE')

CATEGORICAL FREQUENCY: GENDER


### 8.2 Bivariate Relationships
Understanding how two features interact is critical for feature selection in machine learning. Let's look at the relationship between `Age` and `Fare`. Because both are now scaled between 0 and 1, we can easily compare their relative distributions.

In [35]:
# 3. Explore the relationship between two specific features
print("="*60)
print("FEATURE RELATIONSHIP: AGE VS FARE")
print("="*60)

plotter.plot_relationship(
    x='Age',
    y='Fare'
)

FEATURE RELATIONSHIP: AGE VS FARE


### 8.3 Advanced & Multivariate Associations
Let's leverage the power of the `plot_all_associations_heatmap()` function. This will calculate the correlation coefficients across our entire merged dataset, instantly highlighting which features are most strongly correlated with survival rates or passenger class.

In [36]:
# 4. Generate a global correlation/association heatmap
print("="*60)
print("GLOBAL ASSOCIATION HEATMAP")
print("="*60)

plotter.plot_all_associations_heatmap()

GLOBAL ASSOCIATION HEATMAP
